In [1]:
import os, json

In [2]:
import time

In [3]:
from openai import OpenAI
import utils

key_file = 'openai-api-key.txt'
with open(key_file, 'r') as f:
    API_KEY = f.read().strip()

client = OpenAI(  
  api_key=API_KEY
)


In [ ]:
from importlib import reload
reload(utils)

In [5]:
model_name = 'o3-mini'
model_endpoint = utils.model_names_to_endpoints[model_name]
model_endpoint

'o3-mini-2025-01-31'

basic mc questions

In [12]:
data_dir = '../data/final_dataset'
mc_files = ['certamen_mc.json', 'nle_other_questions_2015.json', 'nle_other_questions_2020.json', 'nle_other_questions_2025.json']
mc_files = [os.path.join(data_dir, f) for f in mc_files]

file_to_data = {}
for file in mc_files:
    base_name = os.path.basename(file)
    file_to_data[base_name] = []
    with open(file, 'r') as f:
        file_to_data[base_name] += json.load(f)

    print(base_name, len(file_to_data[base_name]))

certamen_mc.json 315
nle_other_questions_2015.json 171
nle_other_questions_2020.json 173
nle_other_questions_2025.json 130


In [7]:
def construct_mc_user_prompt(q_dict):
    question_text = q_dict['question'] if 'question' in q_dict else q_dict['question_text']
    choices = q_dict['multiple_choice_options']
    choices_text = '\n'.join(choices)
    question_text += '\n' + choices_text
    question_text += '\n' + utils.mc_format_instructions

    #if model_name == 'qwq' and thinking:
    #    question_text += '\n<think>\n'
    return question_text


In [8]:
prompt = construct_mc_user_prompt(file_to_data['certamen_mc.json'][0])
prompt

'What Latin preposition is the root of “country”?\nA: CONTRA\nB: CUM\nC: ULTRA\nAt the end of your response, give the letter of the correct answer as\nAnswer: Letter'

In [9]:
file_to_data['certamen_mc.json'][0]

{'source_name': 'NJCL-Certamen',
 'source_year': 1996,
 'question_id': 'NJCL-Certamen_1996_9a_1',
 'question_format': 'multiple_choice',
 'question_content': 'vocabulary',
 'difficulty': 'unknown',
 'question_language': 'english',
 'answer_language': 'latin',
 'question': 'What Latin preposition is the root of “country”?',
 'multiple_choice_options': ['A: CONTRA', 'B: CUM', 'C: ULTRA'],
 'answers': ['A: CONTRA']}

In [10]:
response = client.responses.create(
  model=model_endpoint,
  instructions = utils.sys_prompt,
  input = prompt,
  #temperature=0.6, 
  #top_p=0.95, 
  #min_p=0,
  #top_k=20
)
print(response.output_text)

The word "country" comes from the Old French word "contrée," which itself derives from the Vulgar Latin "contra," meaning "on the other side" or "in opposition to." This referred to the lands outside or beyond a town. Therefore, the correct answer is:

Answer: A


In [11]:
save_dir = f'../data/model_responses/{model_name}'
if not os.path.exists(save_dir):
    os.makedirs(save_dir)


In [13]:
for filename, data in file_to_data.items():
    print(filename)
    q_id_to_resp = {}
    i = 0
    save_file = os.path.join(save_dir, filename)
    if os.path.exists(save_file):
        with open(save_file, 'r') as f:
            q_id_to_resp = json.load(f)
    for q_dict in data:
        q_id = q_dict['question_id']
        if q_id in q_id_to_resp:
            i += 1
            continue
        prompt = construct_mc_user_prompt(q_dict)


        try:
            response = client.responses.create(
                model=model_endpoint,
                instructions = utils.sys_prompt,
                input = prompt,
            )
            resp = response.output_text
        except:
            print(f'Error on {q_id}')
            print(response)
            # dump this file 
            with open(save_file, 'w') as f:
                json.dump(q_id_to_resp, f, indent=4)
            break
        q_id_to_resp[q_id] = resp

        time.sleep(.05)
        

        if i % 100 == 0:
            # dump this file 
            print(f'  {i} / {len(data)}')
            with open(save_file, 'w') as f:
                json.dump(q_id_to_resp, f, indent=4)
            
        i += 1

    # dump this file 
    with open(save_file, 'w') as f:
        json.dump(q_id_to_resp, f, indent=4)
        


certamen_mc.json
  0 / 315
  100 / 315
  200 / 315
  300 / 315
nle_other_questions_2015.json
  0 / 171
  100 / 171
nle_other_questions_2020.json
  0 / 173
  100 / 173
nle_other_questions_2025.json
  0 / 130
  100 / 130


reading comp questions

In [15]:
import glob

In [16]:
files = glob.glob(os.path.join(data_dir, '*.json'))
question_files = [f for f in files if 'reading_comp_questions' in f]
passage_files = [f for f in files if 'reading_comp_passages' in f]

file_to_data = {}
for file in question_files:
    base_name = os.path.basename(file)
    file_to_data[base_name] = []
    with open(file, 'r') as f:
        file_to_data[base_name] += json.load(f)
    print(base_name, len(file_to_data[base_name]))

nle_reading_comp_questions_2015.json 103
nle_reading_comp_questions_2020.json 103
nle_reading_comp_questions_2025.json 176


In [17]:
question_files

['../data/final_dataset/nle_reading_comp_questions_2015.json',
 '../data/final_dataset/nle_reading_comp_questions_2020.json',
 '../data/final_dataset/nle_reading_comp_questions_2025.json']

In [18]:
# create passage id to passage text dict
passage_id_to_txt = {}
for passage_file in passage_files:
    with open(passage_file, 'r') as f:
        passage_data = json.load(f)

    for p_dict in passage_data:
        id_ = p_dict['passage_id']
        text = p_dict['text']
        passage_id_to_txt[id_] = text

In [19]:
def construct_rc_user_prompt(q_dict):

    passage_id = q_dict['passage_id']
    passage_text = passage_id_to_txt[passage_id]

    question_text = f'{passage_text}\n\n'

    question_text += q_dict['question'] if 'question' in q_dict else q_dict['question_text']
    choices = q_dict['multiple_choice_options']
    choices_text = '\n'.join(choices)
    question_text += '\n' + choices_text
    question_text += '\n' + utils.mc_format_instructions

    #if model_name == 'qwq' and thinking:
    #    question_text += '\n<think>\n'
    return question_text

In [20]:
prompt = construct_rc_user_prompt(file_to_data['nle_reading_comp_questions_2015.json'][0])
prompt

'READ THE REST OF THE STORY AND ANSWER THE QUESTIONS.\n\nTHE STRUGGLE\nVir Germānicus ex forō fugit. Senātor et duo fīliī virum agitant. Senātor virum comprehendit. Senātor cum virō pugnat. Turba pugnam videt et circumvenit. Vir turbam timet. Vir effugere temptat et inter duōs puerōs currit. Vir forte puerōs offendit et in terram dēcidit.\n"Tū fīliōs meōs offendere audēs," senātor clāmat. "Ego tibi supplicium postulō quod fīliōs meōs vulnerās."\n"Pater," ūnus fīlius inquit, "vir Germānicus forte nōs vulnerābat. Nōlī pūnīre virum. Vir est viātor. Potest portāre litterās ad Germāniam."\n"Ita vērō," senātor respondet, "Tū es callidus."\n\n1 fugit = flees\n2 agitant = chase; comprehendit = takes hold of\n3 Turba = A crowd\n4 circumvenit = surrounds; effugere = to escape\n5 currit = runs; forte = accidentally\n6 offendit = bumps into; dēcidit = falls down\n7 audēs = dare\n8 supplicium postulō = ask for the death penalty; vulnerās = you are hurting\n9\n10 Nōlī pūnīre = Don\'t punish; viātor 

In [21]:
for filename, data in file_to_data.items():
    print(filename)
    q_id_to_resp = {}
    i = 0
    save_file = os.path.join(save_dir, filename)
    if os.path.exists(save_file):
        with open(save_file, 'r') as f:
            q_id_to_resp = json.load(f)
    for q_dict in data:
        q_id = q_dict['question_id']
        if q_id in q_id_to_resp:
            i += 1
            continue
        prompt = construct_rc_user_prompt(q_dict)

        try:
            response = client.responses.create(
                model=model_endpoint,
                instructions = utils.sys_prompt,
                input = prompt,
            )
            resp = response.output_text
        except:
            print(f'Error on {q_id}')
            print(response)
            # dump this file 
            with open(save_file, 'w') as f:
                json.dump(q_id_to_resp, f, indent=4)
            break
        q_id_to_resp[q_id] = resp

        time.sleep(.03)
        

        if i % 100 == 0:
            # dump this file 
            print(f'  {i} / {len(data)}')
            with open(save_file, 'w') as f:
                json.dump(q_id_to_resp, f, indent=4)
            
        i += 1

    # dump this file 
    with open(save_file, 'w') as f:
        json.dump(q_id_to_resp, f, indent=4)
        


nle_reading_comp_questions_2015.json
  0 / 103
  100 / 103
nle_reading_comp_questions_2020.json
  0 / 103
  100 / 103
nle_reading_comp_questions_2025.json
  0 / 176
  100 / 176
